# Casey raw data to calibrated dynamic spectra

**Scope:** one single-component burst, ending at fit-ready dynamic spectra. No burst-model fit is performed here.

**Raw intensity authority:** CHIME/FRB `singlebeam_362593221.h5` voltages and DSA-110 `240229aaad_dev_polcal_I.fil`. Accepted NumPy products are read only to reconstruct their reviewed channel-support masks; their intensity samples never enter either dynamic spectrum.

**Dispersion convention:** both bands are referred to 400 MHz at the provisional common anchor of 491.28 pc cm^-3. CHIME/FRB is coherently dedispersed once from the H5 voltage state. DSA-110 starts at 491.211 pc cm^-3 and receives only the non-wrapping fractional correction 491.28 - 491.211 pc cm^-3.

**Interference convention:** the accepted support is authoritative for this run. Additional off-pulse outliers are displayed as diagnostic proposals and are not applied. Casey has no approved additional manual channels.

**Calibration convention:** per-channel location and scale are learned only from off-pulse samples before the burst. Samples after the burst are held out to validate the correction. For CHIME/FRB, fine-channel calibration also removes the repeating polyphase-filterbank scallop. Final spectra retain native band-specific grids.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import os
import platform
import sys
import warnings
from contextlib import redirect_stdout
from decimal import Decimal, ROUND_HALF_EVEN
from pathlib import Path

import h5py
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)
from blimpy import Waterfall

get_ipython().run_line_magic("matplotlib", "inline")

ANALYSIS_ROOT = Path(os.environ.get("ANALYSIS_ROOT", "/workspace"))
RUN_OUTPUT = Path(os.environ.get("RUN_OUTPUT", "/output"))
CONFIG_PATH = Path(os.environ.get(
    "CONFIG_PATH", ANALYSIS_ROOT / "analysis-configs/absolute-dm/casey.json"
))
sys.path.insert(0, str(ANALYSIS_ROOT / "scripts"))

from absolute_dm_voltage import (
    K_DM_S_MHZ2,
    REFERENCE_FREQUENCY_MHZ,
    authoritative_fine_frequency_centres,
    package_dm_argument,
    physical_dm_from_package_coordinate,
    validate_frequency_map,
)
from one_event_hybrid_dm import absolute_crop, peak_time
from baseband_analysis.core.bbdata import BBData
from baseband_analysis.core.dedispersion import K_DM as PACKAGE_K_DM_S_MHZ2
from baseband_analysis.core.dedispersion import coherent_dedisp
from baseband_analysis.core.sampling import _upchannel

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180, "axes.grid": False})
config = json.loads(CONFIG_PATH.read_text())
assert config["event"] == "casey"
assert float(config["geometry"]["reference_frequency_mhz"]) == 400.0
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
(RUN_OUTPUT / "products").mkdir(exist_ok=True)
print(f"analysis root: {ANALYSIS_ROOT}")
print(f"output: {RUN_OUTPUT}")
print(f"Python: {sys.version.split()[0]}; NumPy: {np.__version__}; matplotlib: {matplotlib.__version__}")

In [ ]:
def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def accepted_support(reference: np.ndarray, expected: dict, instrument: str) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        sigma = np.nanstd(reference, axis=1)
    live = np.isfinite(sigma) & (sigma > 0)
    if reference.shape[0] != int(expected["full_grid_rows"]):
        raise RuntimeError(f"{instrument} accepted support row count changed")
    if int(live.sum()) != int(expected["live_count"]):
        raise RuntimeError(f"{instrument} accepted live-channel count changed")
    return live


def robust_z(values: np.ndarray, selected: np.ndarray) -> np.ndarray:
    result = np.full(values.shape, np.nan, dtype=float)
    use = selected & np.isfinite(values)
    center = float(np.median(values[use]))
    scale = float(1.4826 * np.median(np.abs(values[use] - center)))
    if not np.isfinite(scale) or scale <= 0:
        raise RuntimeError("robust scale is not positive")
    result[use] = (values[use] - center) / scale
    return result


def offpulse_calibration(
    data: np.ndarray,
    trusted_live: np.ndarray,
    peak_sample: int,
    guard_samples: int,
) -> dict:
    ntime = data.shape[1]
    left = np.arange(ntime) < peak_sample - guard_samples
    right = np.arange(ntime) > peak_sample + guard_samples
    if left.sum() < 32 or right.sum() < 32:
        raise RuntimeError("insufficient independent off-pulse training and validation samples")
    training = np.asarray(data[:, left], dtype=float)
    validation = np.asarray(data[:, right], dtype=float)
    mean = np.full(data.shape[0], np.nan, dtype=float)
    scale = np.full(data.shape[0], np.nan, dtype=float)
    mean[trusted_live] = np.mean(training[trusted_live], axis=1)
    scale[trusted_live] = np.std(training[trusted_live], axis=1, ddof=1)
    calibration_valid = trusted_live & np.isfinite(mean) & np.isfinite(scale) & (scale > 0)
    if not np.array_equal(calibration_valid, trusted_live):
        bad = np.flatnonzero(trusted_live & ~calibration_valid)
        raise RuntimeError(f"accepted-live calibration failed for rows {bad.tolist()}")
    corrected = np.full(data.shape, np.nan, dtype=np.float32)
    corrected[trusted_live] = (
        (np.asarray(data[trusted_live], dtype=float) - mean[trusted_live, None])
        / scale[trusted_live, None]
    ).astype(np.float32)
    heldout_mean = np.full(data.shape[0], np.nan, dtype=float)
    heldout_scale = np.full(data.shape[0], np.nan, dtype=float)
    heldout_mean[trusted_live] = np.mean(corrected[trusted_live][:, right], axis=1)
    heldout_scale[trusted_live] = np.std(corrected[trusted_live][:, right], axis=1, ddof=1)
    mean_z = robust_z(heldout_mean, trusted_live)
    log_scale_z = robust_z(np.log(heldout_scale), trusted_live)
    proposed_extra = trusted_live & ((np.abs(mean_z) > 5.0) | (np.abs(log_scale_z) > 5.0))
    return {
        "corrected": corrected,
        "mean": mean,
        "scale": scale,
        "heldout_mean": heldout_mean,
        "heldout_scale": heldout_scale,
        "proposed_extra": proposed_extra,
        "training": left,
        "validation": right,
        "guard": ~(left | right),
    }


def profile_peak(data: np.ndarray, live: np.ndarray) -> int:
    values = np.asarray(data[live], dtype=float)
    center = np.nanmedian(values, axis=1)
    sigma = 1.4826 * np.nanmedian(np.abs(values - center[:, None]), axis=1)
    usable = np.isfinite(sigma) & (sigma > 0)
    z = (values[usable] - center[usable, None]) / sigma[usable, None]
    return int(np.nanargmax(np.nanmean(np.clip(z, 0.0, None), axis=0)))


def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    partial = path.with_suffix(path.suffix + ".partial")
    with partial.open("wb") as stream:
        np.savez(stream, **arrays)
    os.replace(partial, path)


def block_mean_rows(data: np.ndarray, frequency: np.ndarray, target_rows: int = 512):
    order = np.argsort(frequency)
    values = np.asarray(data[order], dtype=float)
    freq = np.asarray(frequency[order], dtype=float)
    factor = max(1, values.shape[0] // target_rows)
    usable = values.shape[0] // factor * factor
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        reduced = np.nanmean(values[:usable].reshape(-1, factor, values.shape[1]), axis=1)
        reduced_frequency = np.nanmean(freq[:usable].reshape(-1, factor), axis=1)
    return reduced, reduced_frequency


def mjd_sample_to_unix_ns(mjd: object, sample: float, dt_s: float) -> int:
    seconds = (Decimal(str(mjd)) - Decimal("40587")) * Decimal("86400")
    seconds += Decimal(str(sample)) * Decimal(str(dt_s))
    return int((seconds * Decimal("1000000000")).to_integral_value(rounding=ROUND_HALF_EVEN))

## Input identity and raw-product state

In [ ]:
paths = {key: Path(value) for key, value in config["paths"].items() if key in config["input_sha256"]}
identity_rows = []
for key in ("raw_chime_h5", "accepted_chime_reference", "raw_dsa_filterbank", "accepted_dsa_reference"):
    path = paths[key]
    observed = sha256_file(path)
    expected = config["input_sha256"][key]
    if observed != expected:
        raise RuntimeError(f"{key} SHA-256 mismatch")
    identity_rows.append({
        "role": key,
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": observed,
        "intensity_input": key in {"raw_chime_h5", "raw_dsa_filterbank"},
    })
print(json.dumps(identity_rows, indent=2, sort_keys=True))

## CHIME/FRB: H5 metadata and accepted support

In [ ]:
chime_reference = np.load(paths["accepted_chime_reference"], mmap_mode="r")
chime_live_full = accepted_support(chime_reference, config["chime"]["accepted_support"], "CHIME/FRB")

with h5py.File(paths["raw_chime_h5"], "r") as handle:
    chime_frequency_id = np.asarray(handle["index_map/freq"]["id"], dtype=np.int64)
    chime_coarse_frequency_mhz = np.asarray(handle["index_map/freq"]["centre"], dtype=float)
    chime_row_ctime = np.asarray(handle["time0"]["ctime"], dtype=float)
    chime_row_ctime_offset = np.asarray(handle["time0"]["ctime_offset"], dtype=float)
    chime_raw_dt_s = float(handle.attrs["delta_time"])
    chime_h5_dm = float(handle["tiedbeam_baseband"].attrs.get("DM", 0.0))
    chime_baseband_sha = str(handle.attrs["baseband-analysis_git_sha"])

validate_frequency_map(chime_frequency_id, chime_coarse_frequency_mhz, chime_frequency_id.size)
missing_ids = np.setdiff1d(np.arange(1024), chime_frequency_id)
present_dead_ids = chime_frequency_id[~chime_live_full[chime_frequency_id]]
expected_dead = np.asarray(config["chime"]["accepted_support"]["h5_present_accepted_dead_ids"])
assert np.array_equal(present_dead_ids, expected_dead)
assert missing_ids.size == config["chime"]["accepted_support"]["h5_missing_count"]
assert not np.any(chime_live_full[missing_ids])

status = np.full(1024, 0)
status[missing_ids] = 1
status[present_dead_ids] = 2
status[np.flatnonzero(chime_live_full)] = 3
labels = ["unclassified", "absent from H5", "H5-present accepted-dead", "accepted-live"]
colors = ["0.7", "#8c8c8c", "#d95f02", "#1b9e77"]
fig, ax = plt.subplots(figsize=(9.2, 2.6), constrained_layout=True)
for code in (1, 2, 3):
    use = status == code
    ax.scatter(np.flatnonzero(use), np.full(use.sum(), code), s=8, color=colors[code], label=f"{labels[code]} ({use.sum()})")
ax.set(xlabel="Authoritative CHIME/FRB coarse-channel identifier", ylabel="Support class", yticks=[])
ax.legend(frameon=False, ncol=3, loc="upper center")
plt.show()

print(json.dumps({
    "raw_sample_time_s": chime_raw_dt_s,
    "H5_DM_attribute_package_coordinate": chime_h5_dm,
    "H5_present_channels": int(chime_frequency_id.size),
    "H5_missing_channels": int(missing_ids.size),
    "accepted_live_channels": int(chime_live_full.sum()),
    "H5_present_accepted_dead": int(present_dead_ids.size),
    "manual_additional_channels": config["chime"]["accepted_support"]["manual_bad_channel_ids"],
    "historical_row_sum_replay_applied": False,
    "baseband_analysis_source_sha": chime_baseband_sha,
}, indent=2, sort_keys=True))

## CHIME/FRB: one coherent correction, upchannelization, and 400 MHz placement

In [ ]:
anchor_dm = float(config["chime"]["anchor_dm_pc_cm3"])
upchannel_factor = int(config["chime"]["upchannel_factor"])
bbdata = BBData.from_file(str(paths["raw_chime_h5"]))
raw_voltage = np.asarray(bbdata["tiedbeam_baseband"][:])
package_argument = package_dm_argument(
    anchor_dm,
    0.0,
    package_dispersion_constant=PACKAGE_K_DM_S_MHZ2,
)
coherent_stdout = io.StringIO()
with redirect_stdout(coherent_stdout):
    anchor_voltage = coherent_dedisp(
        bbdata,
        package_argument,
        matrix_in=raw_voltage,
        time_shift=False,
    )
del raw_voltage
gc.collect()

spectrum, _package_frequency, fine_id = _upchannel(
    anchor_voltage,
    freq_id=chime_frequency_id,
    fftsize=2 * upchannel_factor,
    downfreq=2,
)
del anchor_voltage
gc.collect()
expected_fine_id = (
    chime_frequency_id[:, None] * upchannel_factor
    + np.arange(upchannel_factor)[None, :]
).reshape(-1)
fine_id = np.asarray(fine_id, dtype=np.int64)
assert np.array_equal(fine_id, expected_fine_id)
chime_frequency_mhz = authoritative_fine_frequency_centres(
    chime_frequency_id,
    chime_coarse_frequency_mhz,
    upchannel_factor,
)
chime_intensity = np.asarray((np.abs(spectrum[0]) ** 2 + np.abs(spectrum[1]) ** 2).T, dtype=np.float32)
del spectrum
gc.collect()
chime_live = np.repeat(chime_live_full[chime_frequency_id], upchannel_factor)
chime_shift_frequency_mhz = np.repeat(chime_coarse_frequency_mhz, upchannel_factor)
chime_dt_s = chime_raw_dt_s * 2 * upchannel_factor

row_start_ns = np.asarray([
    int(((Decimal(str(a)) + Decimal(str(b))) * Decimal("1000000000")).to_integral_value(rounding=ROUND_HALF_EVEN))
    for a, b in zip(chime_row_ctime, chime_row_ctime_offset, strict=True)
], dtype=np.int64)
block_center_ns = round((2 * upchannel_factor - 1) / 2 * chime_raw_dt_s * 1e9)
fine_start_ns = np.repeat(row_start_ns + block_center_ns, upchannel_factor)
base_ns = int(np.min(fine_start_ns[chime_live]))
row_start_s = (fine_start_ns - base_ns).astype(float) * 1e-9
referred_start_s = row_start_s - K_DM_S_MHZ2 * anchor_dm * (
    chime_shift_frequency_mhz**-2 - REFERENCE_FREQUENCY_MHZ**-2
)
origin_s = float(np.min(referred_start_s[chime_live]))
end_s = float(np.max(referred_start_s[chime_live] + chime_intensity.shape[1] * chime_dt_s))
aligned_time_s = origin_s + np.arange(int(np.ceil((end_s-origin_s)/chime_dt_s))+1) * chime_dt_s
aligned = np.full((chime_intensity.shape[0], aligned_time_s.size), np.nan, dtype=np.float32)
source_sample = np.arange(chime_intensity.shape[1], dtype=float)
for row in np.flatnonzero(chime_live):
    coordinate = (aligned_time_s - referred_start_s[row]) / chime_dt_s
    aligned[row] = np.interp(coordinate, source_sample, chime_intensity[row], left=np.nan, right=np.nan)
del chime_intensity
gc.collect()

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    peak_s, _ = peak_time(aligned, aligned_time_s, chime_dt_s)
width = int(round(float(config["chime"]["window_s"]) / chime_dt_s))
center = int(round((peak_s - aligned_time_s[0]) / chime_dt_s))
crop_start = center - width // 2
crop_stop = crop_start + width
if crop_start < 0 or crop_stop > aligned.shape[1]:
    raise RuntimeError("CHIME/FRB fixed crop reaches a non-wrapping edge")
chime_crop = np.asarray(aligned[:, crop_start:crop_stop], dtype=np.float32)
chime_time_s = aligned_time_s[crop_start:crop_stop]
chime_time0_unix_ns = base_ns + round(chime_time_s[0] * 1e9)
del aligned
gc.collect()

print(json.dumps({
    "target_absolute_DM_pc_cm3": anchor_dm,
    "reference_frequency_MHz": REFERENCE_FREQUENCY_MHZ,
    "H5_package_DM_attribute": chime_h5_dm,
    "physical_H5_coordinate_DM_pc_cm3": physical_dm_from_package_coordinate(chime_h5_dm, package_dispersion_constant=PACKAGE_K_DM_S_MHZ2),
    "coherent_dedispersion_count": 1,
    "coherent_package_argument": package_argument,
    "coherent_package_diagnostic_stdout": coherent_stdout.getvalue().strip(),
    "incoherent_residual_correction_count": 0,
    "upchannel_factor": upchannel_factor,
    "fine_channel_count": int(chime_crop.shape[0]),
    "accepted_live_fine_channels": int(chime_live.sum()),
    "sample_time_us": chime_dt_s * 1e6,
    "nonwrapping_placement": True,
}, indent=2, sort_keys=True))

## CHIME/FRB: off-pulse calibration and interference validation

In [ ]:
chime_peak = profile_peak(chime_crop, chime_live)
chime_guard_samples = int(np.ceil(max(0.001, 6 * float(config["chime"]["reference_pulse_fwhm_s"])) / chime_dt_s))
chime_cal = offpulse_calibration(chime_crop, chime_live, chime_peak, chime_guard_samples)
chime_corrected = chime_cal["corrected"]
chime_proposed = chime_cal["proposed_extra"]

order = np.argsort(chime_frequency_mhz)
freq = chime_frequency_mhz[order]
live = chime_live[order]
raw_mean = chime_cal["mean"][order]
held_mean = chime_cal["heldout_mean"][order]
held_scale = chime_cal["heldout_scale"][order]
proposed = chime_proposed[order]
fold_raw = np.array([np.nanmedian(raw_mean[live][np.arange(live.sum()) % upchannel_factor == k]) for k in range(upchannel_factor)])
fold_raw /= np.nanmean(fold_raw)
held_all = chime_corrected[:, chime_cal["validation"]][order]
fold_after = np.array([np.nanmedian(held_all[live][np.arange(live.sum()) % upchannel_factor == k]) for k in range(upchannel_factor)])

fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), constrained_layout=True)
axes[0,0].plot(freq[live], raw_mean[live], lw=0.35)
axes[0,0].set(xlabel="Frequency (MHz)", ylabel="Training mean power")
axes[0,1].plot(np.arange(upchannel_factor), fold_raw, marker="o", ms=3, label="raw folded response")
axes[0,1].plot(np.arange(upchannel_factor), fold_after, marker="o", ms=3, label="held-out corrected mean")
axes[0,1].axhline(0, color="0.5", lw=0.7)
axes[0,1].set(xlabel="Fine-bin position inside coarse channel", ylabel="Folded response")
axes[0,1].legend(frameon=False)
axes[1,0].plot(freq[live], held_mean[live], lw=0.35)
axes[1,0].axhline(0, color="0.5", lw=0.7)
axes[1,0].scatter(freq[proposed], held_mean[proposed], s=10, color="#d95f02", label=f"diagnostic only ({proposed.sum()})")
axes[1,0].set(xlabel="Frequency (MHz)", ylabel="Held-out corrected mean")
axes[1,0].legend(frameon=False)
axes[1,1].plot(freq[live], held_scale[live], lw=0.35)
axes[1,1].axhline(1, color="0.5", lw=0.7)
axes[1,1].scatter(freq[proposed], held_scale[proposed], s=10, color="#d95f02")
axes[1,1].set(xlabel="Frequency (MHz)", ylabel="Held-out corrected standard deviation")
plt.show()

print(json.dumps({
    "trusted_live_fine_channels": int(chime_live.sum()),
    "approved_additional_exclusions": 0,
    "diagnostic_proposed_extra_fine_channels_not_applied": int(chime_proposed.sum()),
    "offpulse_training_samples": int(chime_cal["training"].sum()),
    "offpulse_validation_samples": int(chime_cal["validation"].sum()),
    "guard_samples": int(chime_cal["guard"].sum()),
    "final_mask_rule": "accepted support only; diagnostic proposals displayed separately",
}, indent=2, sort_keys=True))

## DSA-110: filterbank metadata, exactly-once residual correction, and accepted support

In [ ]:
dsa_reference = np.load(paths["accepted_dsa_reference"], mmap_mode="r")
dsa_live = accepted_support(dsa_reference, config["dsa"]["accepted_support"], "DSA-110")
dsa_reader = Waterfall(str(paths["raw_dsa_filterbank"]), load_data=True)
dsa_raw = np.asarray(dsa_reader.data[:, 0, :], dtype=np.float32).T
dsa_dt_s = float(dsa_reader.header["tsamp"])
dsa_frequency_mhz = float(dsa_reader.header["fch1"]) + float(dsa_reader.header["foff"]) * np.arange(int(dsa_reader.header["nchans"]))
dsa_input_dm = float(config["dsa"]["accepted_reference_dm_pc_cm3"])
dsa_residual_dm = anchor_dm - dsa_input_dm
dsa_crop_start = int(config["dsa"]["raw_crop_start_sample"])
dsa_crop_samples = int(config["dsa"]["crop_samples"])
source_axis = np.arange(dsa_raw.shape[1], dtype=float)
output_offset = np.arange(dsa_crop_samples, dtype=float)
shift_samples = -K_DM_S_MHZ2 * dsa_residual_dm * (
    dsa_frequency_mhz**-2 - REFERENCE_FREQUENCY_MHZ**-2
) / dsa_dt_s
dsa_crop = np.full((dsa_raw.shape[0], dsa_crop_samples), np.nan, dtype=np.float32)
for row, shift in enumerate(shift_samples):
    source_coordinate = dsa_crop_start + output_offset - shift
    dsa_crop[row] = np.interp(source_coordinate, source_axis, dsa_raw[row], left=np.nan, right=np.nan)
del dsa_raw
gc.collect()
if not np.all(np.isfinite(dsa_crop[dsa_live])):
    raise RuntimeError("DSA-110 crop reaches a non-wrapping filterbank edge")
dsa_time_s = np.arange(dsa_crop_samples) * dsa_dt_s
dsa_time0_unix_ns = mjd_sample_to_unix_ns(dsa_reader.header["tstart"], dsa_crop_start, dsa_dt_s)

fig, ax = plt.subplots(figsize=(9.2, 2.6), constrained_layout=True)
ax.scatter(dsa_frequency_mhz[dsa_live], np.ones(dsa_live.sum()), s=5, color="#1b9e77", label=f"accepted-live ({dsa_live.sum()})")
ax.scatter(dsa_frequency_mhz[~dsa_live], np.zeros((~dsa_live).sum()), s=8, color="#d95f02", label=f"accepted-dead ({(~dsa_live).sum()})")
ax.set(xlabel="Authoritative DSA-110 channel center (MHz)", ylabel="Support class", yticks=[])
ax.legend(frameon=False, ncol=2, loc="upper center")
plt.show()

print(json.dumps({
    "input_product_DM_pc_cm3": dsa_input_dm,
    "target_absolute_DM_pc_cm3": anchor_dm,
    "applied_residual_DM_pc_cm3": dsa_residual_dm,
    "residual_correction_count": 1,
    "reference_frequency_MHz": REFERENCE_FREQUENCY_MHZ,
    "fractional_sample_interpolation": True,
    "nonwrapping_placement": True,
    "raw_crop_start_sample": dsa_crop_start,
    "frequency_order_preserved": "direct",
    "sample_time_us": dsa_dt_s * 1e6,
    "accepted_live_channels": int(dsa_live.sum()),
    "manual_additional_channels": config["dsa"]["accepted_support"]["manual_bad_channel_ids"],
}, indent=2, sort_keys=True))

## DSA-110: off-pulse calibration and interference validation

In [ ]:
dsa_peak = profile_peak(dsa_crop, dsa_live)
dsa_guard_samples = int(np.ceil(0.005 / dsa_dt_s))
dsa_cal = offpulse_calibration(dsa_crop, dsa_live, dsa_peak, dsa_guard_samples)
dsa_corrected = dsa_cal["corrected"]
dsa_proposed = dsa_cal["proposed_extra"]

order = np.argsort(dsa_frequency_mhz)
freq = dsa_frequency_mhz[order]
live = dsa_live[order]
proposed = dsa_proposed[order]
raw_mean = dsa_cal["mean"][order]
held_mean = dsa_cal["heldout_mean"][order]
held_scale = dsa_cal["heldout_scale"][order]

fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), constrained_layout=True)
axes[0,0].plot(freq[live], raw_mean[live], lw=0.45)
axes[0,0].set(xlabel="Frequency (MHz)", ylabel="Training mean power")
axes[0,1].plot(freq[live], dsa_cal["scale"][order][live], lw=0.45)
axes[0,1].set(xlabel="Frequency (MHz)", ylabel="Training standard deviation")
axes[1,0].plot(freq[live], held_mean[live], lw=0.45)
axes[1,0].axhline(0, color="0.5", lw=0.7)
axes[1,0].scatter(freq[proposed], held_mean[proposed], s=10, color="#d95f02", label=f"diagnostic only ({proposed.sum()})")
axes[1,0].set(xlabel="Frequency (MHz)", ylabel="Held-out corrected mean")
axes[1,0].legend(frameon=False)
axes[1,1].plot(freq[live], held_scale[live], lw=0.45)
axes[1,1].axhline(1, color="0.5", lw=0.7)
axes[1,1].scatter(freq[proposed], held_scale[proposed], s=10, color="#d95f02")
axes[1,1].set(xlabel="Frequency (MHz)", ylabel="Held-out corrected standard deviation")
plt.show()

print(json.dumps({
    "trusted_live_channels": int(dsa_live.sum()),
    "approved_additional_exclusions": 0,
    "diagnostic_proposed_extra_channels_not_applied": int(dsa_proposed.sum()),
    "offpulse_training_samples": int(dsa_cal["training"].sum()),
    "offpulse_validation_samples": int(dsa_cal["validation"].sum()),
    "guard_samples": int(dsa_cal["guard"].sum()),
    "final_mask_rule": "accepted support only; diagnostic proposals displayed separately",
}, indent=2, sort_keys=True))

## Materialize fit-ready products and provenance

In [ ]:
chime_product = RUN_OUTPUT / "products/chime-dynamic-spectrum.npz"
dsa_product = RUN_OUTPUT / "products/dsa-dynamic-spectrum.npz"
atomic_npz(
    chime_product,
    waterfall=chime_corrected,
    frequency_mhz=chime_frequency_mhz,
    accepted_live=chime_live,
    diagnostic_proposed_extra=chime_proposed,
    sample_time_s=np.asarray(chime_dt_s),
    time0_unix_ns=np.asarray(chime_time0_unix_ns, dtype=np.int64),
    product_dm_pc_cm3=np.asarray(anchor_dm),
    reference_frequency_mhz=np.asarray(REFERENCE_FREQUENCY_MHZ),
)
atomic_npz(
    dsa_product,
    waterfall=dsa_corrected,
    frequency_mhz=dsa_frequency_mhz,
    accepted_live=dsa_live,
    diagnostic_proposed_extra=dsa_proposed,
    sample_time_s=np.asarray(dsa_dt_s),
    time0_unix_ns=np.asarray(dsa_time0_unix_ns, dtype=np.int64),
    product_dm_pc_cm3=np.asarray(anchor_dm),
    input_dm_pc_cm3=np.asarray(dsa_input_dm),
    reference_frequency_mhz=np.asarray(REFERENCE_FREQUENCY_MHZ),
)
provenance = {
    "schema": "faber2026-casey-raw-dynamic-spectra-v1",
    "status": "provisional_fit_input_pending_owner_review",
    "event": "casey",
    "analysis_commit": os.environ.get("ANALYSIS_COMMIT", "unrecorded"),
    "analysis_tree": os.environ.get("ANALYSIS_TREE", "unrecorded"),
    "container_image_digest": os.environ.get("CONTAINER_IMAGE_DIGEST", "unrecorded"),
    "config": {"path": str(CONFIG_PATH), "sha256": sha256_file(CONFIG_PATH)},
    "inputs": identity_rows,
    "dispersion": {
        "absolute_anchor_dm_pc_cm3": anchor_dm,
        "reference_frequency_mhz": REFERENCE_FREQUENCY_MHZ,
        "chime_coherent_correction_count": 1,
        "chime_residual_correction_count": 0,
        "dsa_input_dm_pc_cm3": dsa_input_dm,
        "dsa_residual_dm_pc_cm3": dsa_residual_dm,
        "dsa_residual_correction_count": 1,
        "nonwrapping_fractional_placement": True,
    },
    "support": {
        "chime_accepted_live_coarse": int(chime_live_full.sum()),
        "chime_h5_missing": int(missing_ids.size),
        "chime_h5_present_accepted_dead": int(present_dead_ids.size),
        "chime_diagnostic_extra_not_applied": int(chime_proposed.sum()),
        "dsa_accepted_live": int(dsa_live.sum()),
        "dsa_accepted_dead": int((~dsa_live).sum()),
        "dsa_diagnostic_extra_not_applied": int(dsa_proposed.sum()),
    },
    "outputs": {
        "chime": {"path": str(chime_product), "sha256": sha256_file(chime_product)},
        "dsa": {"path": str(dsa_product), "sha256": sha256_file(dsa_product)},
    },
}
provenance_path = RUN_OUTPUT / "run-provenance.json"
partial = provenance_path.with_suffix(".json.partial")
partial.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n")
os.replace(partial, provenance_path)
print(json.dumps(provenance, indent=2, sort_keys=True))

## Final CHIME/FRB dynamic spectrum

Calibrated H5-derived Stokes I at the provisional common 491.28 pc cm^-3 anchor. Only the accepted support is masked. Time is shown relative to the provisional single-component peak; the product retains its precise 400 MHz-referenced time origin for the later joint fit.

In [ ]:
chime_display, chime_display_frequency = block_mean_rows(chime_corrected, chime_frequency_mhz)
chime_time_ms = (np.arange(chime_corrected.shape[1]) - chime_peak) * chime_dt_s * 1e3
finite = chime_display[np.isfinite(chime_display)]
vmin, vmax = np.percentile(finite, [2, 99.7])
fig, ax = plt.subplots(figsize=(10.2, 5.0), constrained_layout=True)
image = ax.imshow(
    chime_display,
    origin="lower",
    aspect="auto",
    extent=[chime_time_ms[0], chime_time_ms[-1], chime_display_frequency[0], chime_display_frequency[-1]],
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
)
ax.set(xlabel="Time relative to provisional 400 MHz component peak (ms)", ylabel="Frequency (MHz)")
fig.colorbar(image, ax=ax, label="Calibrated intensity (off-pulse standard deviations)")
plt.show()

## Final DSA-110 dynamic spectrum

Calibrated filterbank-derived Stokes I at the same provisional 491.28 pc cm^-3 anchor and 400 MHz reference. The direct filterbank frequency order was preserved during processing and sorted only for display. Only the accepted support is masked.

In [ ]:
dsa_display, dsa_display_frequency = block_mean_rows(dsa_corrected, dsa_frequency_mhz)
dsa_time_ms = (np.arange(dsa_corrected.shape[1]) - dsa_peak) * dsa_dt_s * 1e3
finite = dsa_display[np.isfinite(dsa_display)]
vmin, vmax = np.percentile(finite, [2, 99.7])
fig, ax = plt.subplots(figsize=(10.2, 5.0), constrained_layout=True)
image = ax.imshow(
    dsa_display,
    origin="lower",
    aspect="auto",
    extent=[dsa_time_ms[0], dsa_time_ms[-1], dsa_display_frequency[0], dsa_display_frequency[-1]],
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    interpolation="nearest",
)
ax.set(xlabel="Time relative to provisional 400 MHz component peak (ms)", ylabel="Frequency (MHz)")
fig.colorbar(image, ax=ax, label="Calibrated intensity (off-pulse standard deviations)")
plt.show()